In [0]:
# ============================================================
# Silver — Source 11: ERP Finance Export
#
# Transformations:
#   - Cast invoice_date, due_date to timestamp
#   - Derive total_pence = subtotal_pence + tax_pence (generator bug)
#   - Normalise status, currency, payment_terms
#   - Reject null invoice_number or order_id → quarantine
#   - vendor_id/vendor_name null is valid (generator limitation)
#   - Deduplicate on invoice_number
#
# Source:  bronze.src_11_finance.invoices
# Target:  silver.src_11_finance.invoices
# Quarantine: silver.quarantine.src_11_finance
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window

BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'
TARGET_TABLE = f'{SILVER_CATALOG}.src_11_finance.invoices'
QUARANTINE_TABLE = f'{SILVER_CATALOG}.quarantine.src_11_finance'

VALID_STATUSES = ['issued', 'paid', 'overdue', 'cancelled', 'draft', 'void']

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_11_finance')
print('Silver Source 11 ERP — starting...')


In [0]:
bronze = spark.table(f'{BRONZE_CATALOG}.src_11_finance.invoices')
total = bronze.count()
print(f'Bronze rows: {total}')

# Cast timestamps — mixed formats, use try_cast via to_timestamp
df = bronze \
    .withColumn('invoice_date', F.to_timestamp(F.col('invoice_date'))) \
    .withColumn('due_date',     F.to_timestamp(F.col('due_date')))

# Derive total_pence from components (total_pence is NULL in source)
df = df.withColumn('total_pence',
    F.when(
        F.col('total_pence').isNull(),
        F.col('subtotal_pence') + F.col('tax_pence')
    ).otherwise(F.col('total_pence'))
)

# Normalise
df = df \
    .withColumn('status',        F.lower(F.trim(F.col('status')))) \
    .withColumn('currency',      F.upper(F.trim(F.col('currency')))) \
    .withColumn('payment_terms', F.upper(F.trim(F.col('payment_terms'))))

# Bad rows
bad = df.filter(
    F.col('invoice_number').isNull() |
    F.col('order_id').isNull() |
    F.col('subtotal_pence').isNull() |
    (F.col('subtotal_pence') <= 0) |
    F.col('invoice_date').isNull() |
    ~F.col('status').isin(VALID_STATUSES)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('invoices'))

# Good rows
good = df.filter(
    F.col('invoice_number').isNotNull() &
    F.col('order_id').isNotNull() &
    F.col('subtotal_pence').isNotNull() &
    (F.col('subtotal_pence') > 0) &
    F.col('invoice_date').isNotNull() &
    F.col('status').isin(VALID_STATUSES)
)

w = Window.partitionBy('invoice_number').orderBy(F.col('invoice_date').desc())
good = good.withColumn('_rn', F.row_number().over(w)) \
           .filter(F.col('_rn') == 1).drop('_rn')

bad_count = bad.count()
good_count = good.count()
print(f'Invoices: {total} total → {good_count} clean, {bad_count} quarantined ({bad_count/total*100:.1f}%)')

# Show status distribution
df.groupBy('status').count().orderBy('count', ascending=False).show()

# Write
if spark.catalog.tableExists(TARGET_TABLE):
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(good.alias('s'), 't.invoice_number = s.invoice_number') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    good.write.format('delta').mode('overwrite').saveAsTable(TARGET_TABLE)
print('✅ Written')

# Quarantine
if bad_count > 0:
    bad.select(
        F.lit('src_11_finance').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    ).write.format('delta').mode('append').option('mergeSchema','true').saveAsTable(QUARANTINE_TABLE)
    print(f'✅ {bad_count} quarantined')


In [0]:
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'silver.src_11_finance.invoices: {count} rows')
spark.sql(f"""
    SELECT status, currency, COUNT(*) as cnt, SUM(total_pence) as total
    FROM {TARGET_TABLE}
    GROUP BY status, currency
    ORDER BY cnt DESC
""").show()
